# Table of Contents

#### Imported Packages

In [2]:
import numpy as np
import pandas as pd
from surprise import Dataset, Reader, SVD
# Assuming these are your local utility files
from evaluation import rmse_function 
from preprocessing import train_test
from config import RATING_PARQUET_PATH

In [5]:
def train_svd(train_df):
    # 1. Define the Reader - ensure the scale matches your actual data
    reader = Reader(rating_scale=(1, 5))
    
    # 2. Load the training data
    # Surprise expects exactly [user, item, rating] columns in this order
    data = Dataset.load_from_df(train_df[["user_id", "movie_id", "rating"]], reader)
    trainset = data.build_full_trainset()
    

    # 3. Initialize and train SVD
    model = SVD(n_factors=50, reg_all=0.05, n_epochs=1, verbose=True)
    model.fit(trainset)

    return model

In [ ]:
def predict_svd(model, test_df):
    # 4. Efficient Prediction
    # Convert test_df to the list of tuples format Surprise expects: [(uid, iid, r_ui), ...]
    # This is much faster than manual zipping for large dataframes
    testset_tuples = list(test_df[["user_id", "movie_id", "rating"]].itertuples(index=False, name=None))
    predictions = model.test(testset_tuples)

    # 5. Extract estimated ratings
    # .est is the predicted rating
    preds = np.array([pred.est for pred in predictions])
    return preds
